In [5]:
import requests
import json
from langchain_community.document_loaders import TextLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 设置Kimi API的参数
api_key = "sk-i2L8ou6zNepSe9UPakjNavpcT60OpAGCL4XYJxm0BVFxXBLX"
api_url = "https://api.moonshot.cn/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# 加载和分割文档
def load_and_split_documents(file_path):
    loader = TextLoader(file_path, encoding="utf-8")
    docs = loader.load()
    text_splitter = SemanticChunker(HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))
    documents = text_splitter.split_documents(docs)
    return documents

# 构建向量数据库
def build_vector_db(documents):
    embeddings = HuggingFaceEmbeddings()
    vector_store = FAISS.from_documents(documents, embeddings)
    return vector_store

# 调用Kimi API进行生成
def call_kimi_api(prompt):
    data = {
        "model": "moonshot-v1-8k",
        "messages": [{"role": "user", "content": prompt}]
    }
    response = requests.post(api_url, headers=headers, data=json.dumps(data))
    response_json = response.json()
    return response_json["choices"][0]["message"]["content"]

# RAG查询
def rag_query(question, vector_store):
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    retrieved_docs = retriever.get_relevant_documents(question)
    context = "\n".join([doc.page_content for doc in retrieved_docs])
    prompt = f"根据以下上下文：\n{context}\n问题：{question}\n回答要求：仅使用给定上下文，不确定时回答'暂不了解'。"
    answer = call_kimi_api(prompt)
    return answer

# 主程序
if __name__ == "__main__":
    file_path = "davinci.txt"  # 替换为你的文档路径
    documents = load_and_split_documents(file_path)
    vector_store = build_vector_db(documents)
    
    question = "How old was he when he died?"  # 替换为你的问题
    answer = rag_query(question, vector_store)
    print("回答：", answer)

RuntimeError: Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_tf_utils because of the following error (look up to see its traceback):
Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.